# 🧠 Distracted Driver Detection — Étape 2 : Entraînement du Modèle CNN

Ce notebook entraîne un classifier CNN sur les données préparées par le notebook précédent.

**Pipeline :**
1. Détection GPU/CPU
2. Chargement des données prétraitées (`.npy`) OU directement depuis les images
3. Sélection de l'architecture (CNN Custom / VGG19 / MobileNetV2)
4. Entraînement avec callbacks avancés
5. Évaluation : courbes d'apprentissage + matrice de confusion
6. Sauvegarde du meilleur modèle dans `../Models/`

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"TensorFlow : {tf.__version__}")

In [ ]:
# ─────────────────────────────────────────────
# DÉTECTION GPU / CPU
# ─────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')

print("=" * 55)
print(" RESSOURCES DISPONIBLES")
print("=" * 55)

if gpus:
    print(f"✅ GPU(s) détecté(s) : {len(gpus)}")
    for gpu in gpus:
        print(f"   └── {gpu.name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   Croissance mémoire GPU activée (évite les OOM).")
    MIXED_PRECISION = True
    if MIXED_PRECISION:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("   Mixed Precision (float16) activé — entraînement plus rapide.")
else:
    print(f"⚠️  Aucun GPU. Utilisation CPU ({len(cpus)} unité(s)).")
    MIXED_PRECISION = False

print("=" * 55)

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — MODIFIER ICI
# ─────────────────────────────────────────────

# ┌─────────────────────────────────────────────────────────────────┐
# │  SÉLECTEUR D'ARCHITECTURE                                       │
# │  Options : "CNN"  |  "VGG19"  |  "MobileNetV2"                  │
# └─────────────────────────────────────────────────────────────────┘
ARCHITECTURE = "VGG19"  # ← Changer ici

# Chemins
PROCESSED_DATA_PATH = Path("../1-Dataset-Preparation/data/processed")
MODELS_PATH = Path("../Models")
MODELS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH = Path("logs")
LOGS_PATH.mkdir(exist_ok=True)

# Hyperparamètres
IMG_SIZE     = (64, 64, 3)
NUM_CLASSES  = 10
BATCH_SIZE   = 64   # Augmenter si GPU disponible (128, 256)
EPOCHS       = 50   # EarlyStopping arrêtera avant si nécessaire
LEARNING_RATE = 1e-3
VALIDATION_SPLIT = 0.2
RANDOM_SEED  = 42

MODEL_OUTPUT_NAME = f"distracted_driver_{ARCHITECTURE.lower()}.keras"

CLASS_NAMES = [
    "Conduite normale", "SMS (Droit)", "Téléphone (Droit)",
    "SMS (Gauche)", "Téléphone (Gauche)", "Réglage Radio",
    "En train de Boire", "Se retourner", "Maquillage", "Parler au passager",
]

print(f"Architecture sélectionnée : {ARCHITECTURE}")
print(f"Batch size                 : {BATCH_SIZE}")
print(f"Epochs max                 : {EPOCHS}")
print(f"Taux d'apprentissage       : {LEARNING_RATE}")
print(f"Validation split           : {VALIDATION_SPLIT*100:.0f}%")
print(f"Modèle de sortie           : {MODELS_PATH / MODEL_OUTPUT_NAME}")

In [ ]:
# ─────────────────────────────────────────────
# CHARGEMENT DES DONNÉES
# ─────────────────────────────────────────────
import time

x_path = PROCESSED_DATA_PATH / "X_processed.npy"
y_path = PROCESSED_DATA_PATH / "y_labels.npy"

if not x_path.exists() or not y_path.exists():
    raise FileNotFoundError(
        f"Données prétraitées introuvables dans {PROCESSED_DATA_PATH}.\n"
        "Lancez d'abord le notebook '1-Dataset-Preparation/1_Dataset_Preparation.ipynb'."
    )

print("Chargement des données NumPy...")
t0 = time.time()
X = np.load(x_path)
y = np.load(y_path)
print(f"Données chargées en {time.time() - t0:.1f}s")
print(f"X : {X.shape}  (dtype: {X.dtype})")
print(f"y : {y.shape}  (dtype: {y.dtype})")
print(f"Min / Max pixels : {X.min():.3f} / {X.max():.3f}")

# Répartition des classes
unique, counts = np.unique(y, return_counts=True)
print("\nRépartition des classes :")
for cls, cnt in zip(unique, counts):
    print(f"   c{cls} — {CLASS_NAMES[cls]}: {cnt} images")

In [ ]:
# ─────────────────────────────────────────────
# SPLIT TRAIN / VALIDATION
# ─────────────────────────────────────────────

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"Train : {X_train.shape[0]} images")
print(f"Val   : {X_val.shape[0]} images")

# One-hot encoding
y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val, NUM_CLASSES)

# Pipeline tf.data pour des performances maximales
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))\
    .shuffle(buffer_size=10_000, seed=RANDOM_SEED)\
    .batch(BATCH_SIZE)\
    .prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val_oh))\
    .batch(BATCH_SIZE)\
    .prefetch(AUTOTUNE)

print("\nPipelines tf.data créés avec prefetch.")

In [ ]:
# ─────────────────────────────────────────────
# CONSTRUCTION DU MODÈLE
# ─────────────────────────────────────────────

def build_cnn_custom(input_shape, num_classes):
    """CNN léger entraîné from scratch."""
    model = models.Sequential([
        layers.Input(shape=input_shape),
        # Bloc 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),
        # Bloc 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),
        # Bloc 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),
        # Classificateur
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax', dtype='float32'),
    ], name="CNN_Custom")
    return model


def build_vgg19(input_shape, num_classes):
    """VGG19 pré-entraîné ImageNet + fine-tuning."""
    base = keras.applications.VGG19(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    # Geler les couches de base dans un premier temps
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    # Upsampling pour VGG19 qui préfère des entrées plus grandes
    x = layers.UpSampling2D(size=(3, 3))(inputs)  # 64→192
    x = keras.applications.vgg19.preprocess_input(x * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs, name="VGG19_FineTuned")
    return model, base


def build_mobilenetv2(input_shape, num_classes):
    """MobileNetV2 pré-entraîné — léger, rapide sur CPU."""
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_FineTuned")
    return model, base


# ── Construction selon le sélecteur ──────────────────────────────────────────
base_model = None

if ARCHITECTURE == "CNN":
    model = build_cnn_custom(IMG_SIZE, NUM_CLASSES)
elif ARCHITECTURE == "VGG19":
    model, base_model = build_vgg19(IMG_SIZE, NUM_CLASSES)
elif ARCHITECTURE == "MobileNetV2":
    model, base_model = build_mobilenetv2(IMG_SIZE, NUM_CLASSES)
else:
    raise ValueError(f"Architecture inconnue : {ARCHITECTURE}. Choisir CNN, VGG19 ou MobileNetV2.")

model.summary()

In [ ]:
# ─────────────────────────────────────────────
# COMPILATION ET CALLBACKS
# ─────────────────────────────────────────────

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_save_path = str(MODELS_PATH / MODEL_OUTPUT_NAME)

callbacks = [
    # Arrêt anticipé si la val_loss ne s'améliore plus pendant 10 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    # Réduction du LR si val_loss stagne (patience 5 epochs, division par 5)
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    # Sauvegarde du meilleur checkpoint
    ModelCheckpoint(
        filepath=model_save_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # TensorBoard pour suivre l'entraînement
    TensorBoard(log_dir=str(LOGS_PATH / ARCHITECTURE), histogram_freq=1),
]

print(f"Modèle compilé. Meilleur checkpoint sauvegardé dans : {model_save_path}")
print("Callbacks : EarlyStopping(patience=10), ReduceLROnPlateau(patience=5), ModelCheckpoint")

In [ ]:
# ─────────────────────────────────────────────
# ENTRAÎNEMENT — PHASE 1 (têtes de classification uniquement)
# ─────────────────────────────────────────────
import time

print(f"{'='*60}")
print(f" Entraînement Phase 1 — Architecture : {ARCHITECTURE}")
if base_model is not None:
    print(f" (Base {ARCHITECTURE} gelée — uniquement les couches ajoutées)")
print(f"{'='*60}")

t0 = time.time()
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)
print(f"\nPhase 1 terminée en {(time.time() - t0) / 60:.1f} min.")

In [ ]:
# ─────────────────────────────────────────────
# FINE-TUNING (si modèle pré-entraîné)
# ─────────────────────────────────────────────

if base_model is not None:
    print("\nDéblocage partiel du modèle de base pour fine-tuning...")

    # Dégeler les 30 dernières couches du modèle de base
    base_model.trainable = True
    FINE_TUNE_AT = max(0, len(base_model.layers) - 30)
    for layer in base_model.layers[:FINE_TUNE_AT]:
        layer.trainable = False

    trainable_count = sum(1 for l in base_model.layers if l.trainable)
    print(f"   Couches dégelées : {trainable_count} / {len(base_model.layers)}")

    # Re-compilation avec un LR plus bas
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    ft_callbacks = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-8, verbose=1),
        ModelCheckpoint(filepath=model_save_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    ]

    t0 = time.time()
    history_ft = model.fit(
        train_ds,
        epochs=20,
        validation_data=val_ds,
        callbacks=ft_callbacks,
        verbose=1
    )
    print(f"Fine-tuning terminé en {(time.time() - t0) / 60:.1f} min.")
else:
    history_ft = None
    print("(CNN custom — pas de fine-tuning applicable)")

In [ ]:
# ─────────────────────────────────────────────
# VISUALISATION : COURBES D'APPRENTISSAGE
# ─────────────────────────────────────────────

def plot_history(hist_list, title_suffix=""):
    all_acc  = []
    all_val_acc = []
    all_loss = []
    all_val_loss = []
    for h in hist_list:
        if h is None:
            continue
        all_acc.extend(h.history.get('accuracy', []))
        all_val_acc.extend(h.history.get('val_accuracy', []))
        all_loss.extend(h.history.get('loss', []))
        all_val_loss.extend(h.history.get('val_loss', []))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Courbes d'entraînement — {ARCHITECTURE}{title_suffix}", fontsize=14, fontweight='bold')

    epochs_range = range(1, len(all_acc) + 1)

    ax1.plot(epochs_range, all_acc, 'b-', label='Train Accuracy')
    ax1.plot(epochs_range, all_val_acc, 'r-', label='Val Accuracy')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_facecolor('#f8f9fa')

    ax2.plot(epochs_range, all_loss, 'b-', label='Train Loss')
    ax2.plot(epochs_range, all_val_loss, 'r-', label='Val Loss')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor('#f8f9fa')

    plt.tight_layout()
    save_path = LOGS_PATH / f"training_curves_{ARCHITECTURE}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Courbes sauvegardées : {save_path}")

plot_history([history, history_ft])

In [ ]:
# ─────────────────────────────────────────────
# ÉVALUATION : MATRICE DE CONFUSION
# ─────────────────────────────────────────────

print("Chargement du meilleur modèle sauvegardé...")
best_model = keras.models.load_model(model_save_path)

# Prédictions sur le jeu de validation
y_pred_probs = best_model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Rapport textuel
print("\n" + "="*60)
print(" RAPPORT DE CLASSIFICATION")
print("="*60)
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

# Matrice de confusion
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
)
ax.set_xlabel('Classe Prédite', fontsize=12)
ax.set_ylabel('Classe Réelle', fontsize=12)
ax.set_title(f'Matrice de Confusion — {ARCHITECTURE}', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = LOGS_PATH / f"confusion_matrix_{ARCHITECTURE}.png"
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Matrice de confusion sauvegardée : {cm_path}")

## ✅ Entraînement Terminé

| Fichier généré | Description |
|---|---|
| `../Models/distracted_driver_vgg19.keras` | Meilleur checkpoint du modèle |
| `logs/training_curves_VGG19.png` | Courbes accuracy & loss |
| `logs/confusion_matrix_VGG19.png` | Matrice de confusion |

➡️ **Étape suivante** : Démarrer l'API FastAPI avec `cd ../3-Inference-API && uvicorn api:app --reload`